# 05 — Integración: reconocimiento visual + recomendación de draft
**Proyecto:** Recomendador de draft de Brawl Stars con reconocimiento visual de brawlers

**Objetivo:** unir los dos módulos en una sola función. Recibe la imagen del retrato de un brawler y el rol (`aliado` o `enemigo`), identifica al brawler con el clasificador del notebook 04 y devuelve recomendaciones calculadas con las tablas del notebook 03.

**Regla de seguridad:** si el clasificador no está suficientemente seguro, el sistema **no recomienda**. Pide confirmar el brawler. Así un error visual no produce una recomendación equivocada sin que nadie lo note.

**Requisito de consistencia:** las funciones `caracteristicas`, `wilson_lower`, `match_arrays`, `build_tables` y `recomendar` son copias exactas de los notebooks 03 y 04. Si cambias una en esos notebooks, cámbiala también aquí.

## 1. Configuración

In [ ]:
DATA_DIR = "/content/data"           # salida de los notebooks 02, 03 y 04
UMBRAL_CONFIANZA = 0.60              # por debajo, el sistema pide confirmar en lugar de recomendar
MIN_MATCHES = 100                    # mismo valor que en el notebook 03
TOP_K = 5
Z = 1.96

## 2. Carga de artefactos
Cargamos el modelo visual, la lista de clases, las tablas globales de partidas y la tabla larga (para filtros por mapa y modo).

In [ ]:
import io
import numpy as np
import pandas as pd
import joblib
import requests
from PIL import Image

artefactos = joblib.load(f"{DATA_DIR}/clasificador_visual.joblib")
modelo = artefactos["modelos"]["Regresión logística"]
clases = artefactos["clases"]
IMG_SIZE = artefactos["img_size"]
assert list(modelo.classes_) == list(range(len(clases))), "Las clases del modelo no coinciden con la lista guardada"
print("Clases:", len(clases), "| IMG_SIZE:", IMG_SIZE)
if artefactos.get("evaluacion_optimista"):
    print("AVISO: el clasificador se evaluó de forma optimista (ver notebook 04).")

tablas_global = {
    "syn": pd.read_parquet(f"{DATA_DIR}/tabla_sinergia.parquet"),
    "counter": pd.read_parquet(f"{DATA_DIR}/tabla_counter.parquet"),
    "base": pd.read_csv(f"{DATA_DIR}/tabla_base.csv").set_index("brawler"),
}
candidatos = pd.read_csv(f"{DATA_DIR}/selected_brawlers.csv")["brawler"].tolist()
long = pd.read_parquet(f"{DATA_DIR}/brawl_long.parquet")
print("Tablas de sinergia:", len(tablas_global["syn"]), "| counters:", len(tablas_global["counter"]))
print("Candidatos:", len(candidatos))

## 3. Características visuales
Copia exacta de la función del notebook 04. Convierte cualquier imagen en el mismo vector que vio el clasificador.

In [ ]:
from skimage.feature import hog

def caracteristicas(img):
    pequena = img.resize((IMG_SIZE, IMG_SIZE))
    arr = np.asarray(pequena)
    hist = np.concatenate([
        np.histogram(arr[..., c], bins=8, range=(0, 256))[0] for c in range(3)
    ]).astype(np.float32)
    hist /= hist.sum()
    gris = np.asarray(pequena.convert("L")) / 255.0
    descriptor = hog(gris, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2))
    return np.concatenate([hist, descriptor])

def cargar_imagen(fuente):
    """Acepta una ruta de archivo, bytes o un objeto PIL, y devuelve RGB sobre fondo gris."""
    if isinstance(fuente, Image.Image):
        img = fuente
    elif isinstance(fuente, (bytes, bytearray)):
        img = Image.open(io.BytesIO(fuente))
    else:
        img = Image.open(fuente)
    img = img.convert("RGBA")
    fondo = Image.new("RGBA", img.size, (128, 128, 128, 255))
    fondo.alpha_composite(img)
    return fondo.convert("RGB")

## 4. Identificación del brawler
Usamos las probabilidades de la regresión logística. Devolvemos el brawler más probable, su probabilidad y los tres primeros, para que se pueda revisar cuando el modelo duda.

In [ ]:
def identificar(imagen):
    x = caracteristicas(cargar_imagen(imagen)).reshape(1, -1)
    proba = modelo.predict_proba(x)[0]
    orden = np.argsort(proba)[::-1]
    top3 = [(clases[i], round(float(proba[i]), 3)) for i in orden[:3]]
    nombre, confianza = top3[0]
    return nombre, confianza, top3

## 5. Prueba de cableado con el catálogo
Comprobamos que la identificación funciona con los retratos oficiales de BrawlAPI. Estas imágenes (`imageUrl`) se usaron para entrenar, así que este bloque sirve para verificar que el cableado es correcto, **no para medir la precisión**. La medición honesta está en el notebook 04.

In [ ]:
r = requests.get("https://api.brawlapi.com/v1/brawlers", timeout=20)
r.raise_for_status()
por_nombre = {b["name"].upper(): b for b in r.json()["list"]}

aciertos, filas = 0, []
for nombre in clases:
    url = por_nombre[nombre]["imageUrl"]
    img = Image.open(io.BytesIO(requests.get(url, timeout=20).content))
    pred, conf, _ = identificar(img)
    aciertos += int(pred == nombre)
    filas.append({"real": nombre, "predicho": pred, "confianza": conf})

prueba = pd.DataFrame(filas)
print(f"Aciertos en retratos de entrenamiento: {aciertos}/{len(clases)}")
prueba[prueba["real"] != prueba["predicho"]]

## 6. Motor de recomendación
Copia exacta del notebook 03: cota de Wilson, tablas por partidas y función `recomendar`.

In [ ]:
def wilson_lower(victorias, partidas, z=Z):
    p = victorias / partidas
    n = partidas
    centro = p + z**2 / (2 * n)
    margen = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return (centro - margen) / (1 + z**2 / n)

def match_arrays(lg):
    t = (lg.groupby(["match_id", "side"])
           .agg(b=("brawler", list), win=("win", "max"))
           .reset_index())
    t0 = t[t["side"] == 0].set_index("match_id")
    t1 = t[t["side"] == 1].set_index("match_id")
    idx = t0.index.intersection(t1.index)
    t0, t1 = t0.loc[idx], t1.loc[idx]
    A0 = np.array(t0["b"].tolist())
    A1 = np.array(t1["b"].tolist())
    w0 = t0["win"].to_numpy().astype(int)
    return A0, A1, w0

def build_tables(lg):
    A0, A1, w0 = match_arrays(lg)
    base = lg.groupby("brawler")["win"].agg(partidas="size", victorias="sum")
    base["wr"] = base["victorias"] / base["partidas"]

    pares = []
    for team, w in [(A0, w0), (A1, 1 - w0)]:
        for i, j in [(0, 1), (0, 2), (1, 2)]:
            pares.append(pd.DataFrame({"brawler": team[:, i], "aliado": team[:, j], "win": w}))
            pares.append(pd.DataFrame({"brawler": team[:, j], "aliado": team[:, i], "win": w}))
    syn = (pd.concat(pares, ignore_index=True)
             .groupby(["brawler", "aliado"])
             .agg(partidas=("win", "size"), victorias=("win", "sum"))
             .reset_index())

    cont = []
    for i in range(3):
        for j in range(3):
            cont.append(pd.DataFrame({"brawler": A0[:, i], "rival": A1[:, j], "win": w0}))
            cont.append(pd.DataFrame({"brawler": A1[:, j], "rival": A0[:, i], "win": 1 - w0}))
    counter = (pd.concat(cont, ignore_index=True)
                 .groupby(["brawler", "rival"])
                 .agg(partidas=("win", "size"), victorias=("win", "sum"))
                 .reset_index())
    return {"base": base, "syn": syn, "counter": counter, "n_partidas": len(w0)}

def recomendar(brawler, rol, tablas, top_k=TOP_K, min_partidas=MIN_MATCHES):
    base = tablas["base"]
    if rol == "aliado":
        df = tablas["syn"][tablas["syn"]["brawler"] == brawler].rename(columns={"aliado": "candidato"})
    elif rol == "enemigo":
        df = tablas["counter"][tablas["counter"]["rival"] == brawler].rename(columns={"brawler": "candidato"})
    else:
        raise ValueError("rol debe ser 'aliado' o 'enemigo'")

    df = df[df["candidato"].isin(candidatos) & (df["candidato"] != brawler) & (df["partidas"] >= min_partidas)].copy()
    if df.empty:
        return pd.DataFrame(columns=["candidato", "wr", "partidas", "wilson_lb", "dif_vs_base"])
    df["wr"] = df["victorias"] / df["partidas"]
    df["wilson_lb"] = wilson_lower(df["victorias"], df["partidas"])
    df["dif_vs_base"] = df["wr"] - df["candidato"].map(base["wr"])
    out = df.sort_values("wilson_lb", ascending=False).head(top_k)
    return out[["candidato", "wr", "partidas", "wilson_lb", "dif_vs_base"]].reset_index(drop=True).round(3)

_cache_tablas = {}
def tablas_para(mapa=None, modo=None):
    """Global si no hay filtros; si los hay, recalcula sobre el subconjunto (con caché)."""
    if mapa is None and modo is None:
        return tablas_global
    clave = (mapa, modo)
    if clave not in _cache_tablas:
        sub = long
        if mapa is not None:
            sub = sub[sub["map"] == mapa]
        if modo is not None:
            sub = sub[sub["mode"] == modo]
        if sub.empty:
            raise ValueError(f"No hay partidas para mapa={mapa}, modo={modo}")
        _cache_tablas[clave] = build_tables(sub)
    return _cache_tablas[clave]

## 7. Función integrada
`analizar_draft` hace el flujo completo:
1. Identifica el brawler en la imagen.
2. Si la confianza es menor que `UMBRAL_CONFIANZA`, no recomienda y devuelve las tres opciones más probables para confirmar.
3. Si está en el alcance, calcula las recomendaciones para el rol, con el mapa y modo indicados si se dan.

In [ ]:
def analizar_draft(imagen, rol, mapa=None, modo=None):
    nombre, confianza, top3 = identificar(imagen)
    print(f"Brawler detectado: {nombre} (confianza {confianza:.2f})")
    print("Alternativas:", top3[1:])

    if confianza < UMBRAL_CONFIANZA:
        print("Confianza insuficiente: confirma el brawler antes de recomendar.")
        return {"brawler": nombre, "confianza": confianza, "recomendaciones": None, "estado": "confirmar"}

    if nombre not in candidatos:
        print("El brawler no está en el alcance inicial de 20 brawlers.")
        return {"brawler": nombre, "confianza": confianza, "recomendaciones": None, "estado": "fuera_de_alcance"}

    tablas = tablas_para(mapa, modo)
    recs = recomendar(nombre, rol, tablas)
    contexto = f"mapa={mapa or 'todos'}, modo={modo or 'todos'}, partidas={tablas['n_partidas']:,}"
    print(f"Rol: {rol} | {contexto}")
    if recs.empty:
        print("Sin recomendaciones con el mínimo de partidas configurado.")
    else:
        print(recs.to_string(index=False))
    return {"brawler": nombre, "confianza": confianza, "recomendaciones": recs, "estado": "ok"}

## 8. Demostración
Probamos el flujo completo con retratos del catálogo. Usamos `imageUrl` por las mismas razones del bloque 5: sirven para comprobar el flujo, no para medir precisión.

In [ ]:
for nombre in ["AMBER", "SPIKE", "MR. P"]:
    img = Image.open(io.BytesIO(requests.get(por_nombre[nombre]["imageUrl"], timeout=20).content))
    print(f"\n=== Imagen de {nombre} | rol: aliado ===")
    analizar_draft(img, rol="aliado")
    print(f"\n=== Imagen de {nombre} | rol: enemigo, mapa 'Kaboom Canyon' ===")
    analizar_draft(img, rol="enemigo", mapa="Kaboom Canyon")

## 9. Imagen propia (opcional)
Sube una captura recortada del retrato de un brawler. Para resultados fiables, el recorte debe contener solo el retrato, como en el draft. Si el recorte incluye otros elementos, el clasificador puede equivocarse.

In [ ]:
try:
    from google.colab import files
    subidos = files.upload()
    for nombre_archivo, contenido in subidos.items():
        print(f"\n=== {nombre_archivo} ===")
        analizar_draft(contenido, rol="aliado")
except ImportError:
    print("Este bloque requiere Google Colab. Fuera de Colab, pasa una ruta con analizar_draft('ruta.png', rol='aliado').")

## 10. Resumen del sistema
Registramos qué artefactos y configuración usa el sistema, para documentarlo en el informe.

In [ ]:
resumen = {
    "clasificador": "Regresión logística sobre histograma de color + HOG",
    "clases": len(clases),
    "IMG_SIZE": IMG_SIZE,
    "umbral_confianza": UMBRAL_CONFIANZA,
    "min_partidas": MIN_MATCHES,
    "partidas_globales": tablas_global["base"]["partidas"].sum() // 6,
    "evaluacion_optimista_clasificador": artefactos.get("evaluacion_optimista"),
}
for k, v in resumen.items():
    print(f"{k}: {v}")

### Qué debes poder explicar en la presentación
- Por qué el sistema no recomienda cuando el clasificador duda: un error visual se convertiría en una recomendación equivocada sin que el usuario lo note.
- Por qué las funciones están duplicadas en lugar de importadas: en Colab cada notebook es independiente; la consistencia se garantiza copiando exactamente el mismo código.
- Qué limitaciones tiene la integración: el clasificador solo reconoce retratos estándar, el recomendador describe correlaciones históricas y el mapa y modo filtrados pueden tener pocas partidas.
- Qué haría falta para pasar de prototipo a producto: más imágenes reales de captura, un modelo entrenado con más datos y una evaluación con usuarios.